In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
import joblib 

In [2]:
df = pd.read_csv("diabetes_final.csv")
df.head()

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


In [3]:
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
print(df.isnull().sum())
print(df["diabetes"].value_counts())
print(df["diabetes"].value_counts(normalize=True))
print(df.duplicated().sum())

(100000, 9)
['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes']
gender                  object
age                    float64
hypertension             int64
heart_disease            int64
smoking_history         object
bmi                    float64
HbA1c_level            float64
blood_glucose_level      int64
diabetes                 int64
dtype: object
gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blood_glucose_level    0
diabetes               0
dtype: int64
diabetes
0    91500
1     8500
Name: count, dtype: int64
diabetes
0    0.915
1    0.085
Name: proportion, dtype: float64
3854


In [4]:
print(df["gender"].unique())
print(df["smoking_history"].unique())
print(df.describe(include="all"))

['Female' 'Male' 'Other']
['never' 'No Info' 'current' 'former' 'ever' 'not current']
        gender            age  hypertension  heart_disease smoking_history  \
count   100000  100000.000000  100000.00000  100000.000000          100000   
unique       3            NaN           NaN            NaN               6   
top     Female            NaN           NaN            NaN         No Info   
freq     58552            NaN           NaN            NaN           35816   
mean       NaN      41.885856       0.07485       0.039420             NaN   
std        NaN      22.516840       0.26315       0.194593             NaN   
min        NaN       0.080000       0.00000       0.000000             NaN   
25%        NaN      24.000000       0.00000       0.000000             NaN   
50%        NaN      43.000000       0.00000       0.000000             NaN   
75%        NaN      60.000000       0.00000       0.000000             NaN   
max        NaN      80.000000       1.00000       1.0000

In [5]:
duplicate_rows = df[df.duplicated(keep=False)].sort_values(
    by=df.columns.tolist()
)

duplicate_rows.head(20)

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
47708,Female,2.0,0,0,No Info,27.32,5.0,158,0
59468,Female,2.0,0,0,No Info,27.32,5.0,158,0
62073,Female,2.0,0,0,No Info,27.32,6.0,85,0
67439,Female,2.0,0,0,No Info,27.32,6.0,85,0
23617,Female,2.0,0,0,No Info,27.32,6.0,145,0
67234,Female,2.0,0,0,No Info,27.32,6.0,145,0
90685,Female,2.0,0,0,No Info,27.32,6.2,145,0
97294,Female,2.0,0,0,No Info,27.32,6.2,145,0
46785,Female,2.0,0,0,No Info,27.32,6.5,155,0
89701,Female,2.0,0,0,No Info,27.32,6.5,155,0


In [6]:
df = df.drop_duplicates().reset_index(drop=True)

In [7]:
print("After:", df.shape)


After: (96146, 9)


In [8]:
feature_columns = [
    "gender",
    "age",
    "hypertension",
    "heart_disease",
    "smoking_history",
    "bmi",
    "HbA1c_level",
    "blood_glucose_level"
]

conflicts = (
    df.groupby(feature_columns)["diabetes"]
      .nunique()
)

print("Conflicting feature combinations:", (conflicts > 1).sum())

Conflicting feature combinations: 91


In [9]:
print(df["diabetes"].value_counts())
print(df["diabetes"].value_counts(normalize=True))

diabetes
0    87664
1     8482
Name: count, dtype: int64
diabetes
0    0.91178
1    0.08822
Name: proportion, dtype: float64


In [10]:
conflict_groups = (
    df.groupby(feature_columns)["diabetes"]
      .agg(["nunique", "count"])
      .reset_index()
)

conflict_groups = conflict_groups[
    conflict_groups["nunique"] > 1
]

print(conflict_groups)
print("Number of conflicting groups:", len(conflict_groups))

       gender   age  hypertension  heart_disease smoking_history    bmi  \
3347   Female   6.0             0              0         No Info  27.32   
15968  Female  28.0             0              0         No Info  27.32   
23063  Female  37.0             0              0         No Info  27.32   
25545  Female  40.0             0              0         No Info  27.32   
28379  Female  43.0             0              0           never  27.32   
...       ...   ...           ...            ...             ...    ...   
94322    Male  80.0             0              0         No Info  27.32   
94380    Male  80.0             0              0         No Info  27.32   
94388    Male  80.0             0              0         No Info  27.32   
94500    Male  80.0             0              0         current  27.32   
95100    Male  80.0             0              0           never  27.32   

       HbA1c_level  blood_glucose_level  nunique  count  
3347           6.6                  130  

In [11]:
conflict_keys = conflict_groups[feature_columns]

conflicting_rows = df.merge(
    conflict_keys,
    on=feature_columns,
    how="inner"
).sort_values(feature_columns)

print(conflicting_rows.to_string(index=False))

gender  age  hypertension  heart_disease smoking_history   bmi  HbA1c_level  blood_glucose_level  diabetes
Female  6.0             0              0         No Info 27.32          6.6                  130         1
Female  6.0             0              0         No Info 27.32          6.6                  130         0
Female 28.0             0              0         No Info 27.32          6.0                  140         0
Female 28.0             0              0         No Info 27.32          6.0                  140         1
Female 37.0             0              0         No Info 27.32          6.1                  155         0
Female 37.0             0              0         No Info 27.32          6.1                  155         1
Female 40.0             0              0         No Info 27.32          5.8                  126         1
Female 40.0             0              0         No Info 27.32          5.8                  126         0
Female 43.0             0            

In [12]:
feature_columns = [
    "gender",
    "age",
    "hypertension",
    "heart_disease",
    "smoking_history",
    "bmi",
    "HbA1c_level",
    "blood_glucose_level"
]

conflict_counts = (
    df.groupby(feature_columns)["diabetes"]
      .nunique()
)

conflict_keys = conflict_counts[conflict_counts > 1].index

row_keys = pd.MultiIndex.from_frame(df[feature_columns])

df_model = df.loc[
    ~row_keys.isin(conflict_keys)
].reset_index(drop=True)

print("Original rows:", len(df))
print("Rows after removing conflicting groups:", len(df_model))
print("Rows removed:", len(df) - len(df_model))

Original rows: 96146
Rows after removing conflicting groups: 95964
Rows removed: 182


In [13]:
remaining_conflicts = (
    df_model.groupby(feature_columns)["diabetes"]
            .nunique()
)

print(
    "Remaining conflicting groups:",
    (remaining_conflicts > 1).sum()
)

print("Remaining exact duplicates:", df_model.duplicated().sum())

Remaining conflicting groups: 0
Remaining exact duplicates: 0


In [14]:
features = [
    "gender",
    "age",
    "hypertension",
    "heart_disease",
    "smoking_history",
    "bmi",
    "HbA1c_level",
    "blood_glucose_level"
]

X = df_model[features]
y = df_model["diabetes"]

In [15]:
X.shape,y.shape

((95964, 8), (95964,))

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (76771, 8)
X_test: (19193, 8)
y_train: (76771,)
y_test: (19193,)


In [17]:
categorical_features = [
    "gender",
    "smoking_history"
]
numeric_features = [
    "age",
    "hypertension",
    "heart_disease",
    "bmi",
    "HbA1c_level",
    "blood_glucose_level"
]

In [18]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

In [19]:
diabetes_model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        )
    )
])

diabetes_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:

y_pred = diabetes_model.predict(X_test)
y_probability = diabetes_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1-score:", f1_score(y_test, y_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, y_probability))
print("PR-AUC:", average_precision_score(y_test, y_probability))

print("\nConfusion matrix:")b
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.9695201375501485
Precision: 0.9417946645109135
Recall: 0.6942789034564958
F1-score: 0.7993138936535163
ROC-AUC: 0.9649690866027655
PR-AUC: 0.8593515919349732

Confusion matrix:
[[17443    72]
 [  513  1165]]

Classification report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     17515
           1       0.94      0.69      0.80      1678

    accuracy                           0.97     19193
   macro avg       0.96      0.85      0.89     19193
weighted avg       0.97      0.97      0.97     19193



In [22]:
import sys
import numpy
import pandas
import sklearn
import joblib

print("Python:", sys.version)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)

Python: 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
NumPy: 2.3.1
Pandas: 2.3.1
Scikit-learn: 1.7.2
Joblib: 1.5.2


In [21]:
# Save the trained model
joblib.dump(diabetes_model, "diabetes_model.pkl")

['diabetes_model.pkl']